In [ ]:
# Core numerics and data handling
import numpy as np
import pandas as pd

# PyTorch (deep learning)
import torch
import torch.nn as nn

# Preprocessing
# Optimization (MILP/LP)
import pulp
# Plotting
import matplotlib.pyplot as plt
# optimization + integration


In [ ]:
DATA_CSV          = "week.csv"
FREQ_PROFILE_CSV  = "gpu_profiling_results_real_azure.csv"
POOL_PROFILE_CSV  = "merged_window (1).csv"



# IT POWER polynomial
a4_poly, a3_poly, a2_poly, a1_poly, a0_poly = 

# GPU temp 
alpha, beta, gamma = 

# Cooling constraints

# TI_max limits inlet temperature.
# Tc_in_max limits return/hot-aisle temperature.


# Initial cooling control 


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load LSTMs
cluster_token_model.load_state_dict(torch.load("token_forecast_model.pth", map_location=device))
cluster_token_model.eval()
pool_token_model.load_state_dict(torch.load("multi_step_lstm_token_model.pth", map_location=device))
pool_token_model.eval()
# ~>>>>>>>>>>>>>>>>>>>>
# Per-job profilin g (freqs)
# ==========================



In [ ]:
def cluster_level_mpc_profiled(token_forecast_30min: float):
    L_like_5min = token_forecast_30min / (CLUSTER_WINDOW_SEC / PROFILE_WINDOW_SEC) # CLUSTER WINDOW / 6 = 5 MINUTES 
    params = pick_tp_params_for_load(L_like_5min) # This searches your profiling database.
    C8_5min = max(1.0, params["C8"])
    C8_30min = C8_5min * (CLUSTER_WINDOW_SEC / PROFILE_WINDOW_SEC)
    servers_needed = int(np.ceil(token_forecast_30min / C8_30min))
    servers = max(1, min(servers_needed, max_servers))
    return servers, 

# ===== Pool MILP =====
def pool_level_mpc_tokenload_profiled(pred_matrix: np.ndarray, total_gpus: int):
    H = int(pred_matrix.shape[0]) # 
    params = pick_tp_params_for_load(L_like_5min)
    C2, C4, C8 = params["C2"]*H, params["C4"]*H, params["C8"]*H # Convert capacity from: ONE WINDOW TO ALL WINDOW 
    P2, P4, P8 = params["P2"]*H, params["P4"]*H, params["P8"]*H
    T2, T4, T8 = params["T2"], params["T4"], params["T8"]
    prob += tp2*P2 + tp4*P4 + tp8*P8
    prob += 2*tp2 + 4*tp4 + 8*tp8 <= max(0, int(total_gpus))
    prob += tp2*C2 + tp4*C4 + tp8*C8 >= L_total
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    return int(tp2.value() or 0), int(tp4.value() or 0), int(tp8.value() or 0)

def instance_level_mpc_profile_lat_poly_power(job_class: str, ctx_tokens: int, T_C: float, u_s: float):
    cand_freqs = AVAILABLE_FREQS if len(AVAILABLE_FREQS)>0 else
    metrics = {}
    for f in cand_freqs:
        L_prof = float(row["Latency"])
        T_prof = float(row["Temperature"])
        P_est  = poly_power(f, u_s)
        T_gpu  = alpha*T_C + beta*P_est + gamma
        metrics[int(f)] = {"P":P_est, "L":L_prof, "Tprof":T_prof, "Tgpu":T_gpu}

    prob = pulp.LpProblem("InstanceMPC_ProfileLatency_PolyPower", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("f", list(metrics.keys()), cat=pulp.LpBinary)
    prob += pulp.lpSum(metrics[f]["P"] * x[f] for f in metrics)
    prob += pulp.lpSum(x[f] for f in metrics) == 1
    lat_cap = float(LAT_THRESH_S.get(job_class, LAT_THRESH_S["medium"]))
    prob += pulp.lpSum(metrics[f]["L"] * x[f] for f in metrics) <= lat_cap
    prob.solve(pulp.PULP_CBC_CMD(msg=False))

def full_rack_ode_MPC_util(t, T, N, rho, cp, VC, VH, X, T_HPCU, Q_HPCU, Q_L_total, P_IT_arr):
    dTdt = np.zeros_like(T)
        dTdt[idx] = (1/(rho*cp*VC))*(Q_H*rho*cp*T_HPCU + Q_OC_prev*rho*cp*T_C_up + Q_L*rho*cp*T_H
                                     - Q_OC*rho*cp*T_C - Q_S*rho*cp*T_C)
        dTdt[idx+2] = (1/(rho*cp*VH))*(Q_S*rho*cp*T_S - Q_L*rho*cp*T_H + Q_OH_prev*rho*cp*T_H_dn - Q_OH_curr*rho*cp*T_H)
    return dTdt